### Practical Exercise 3B: Uniform Cost Search
#### Finding the shortest route from Nairobi to Kisumu

In this exercise, we are using Uniform Cost Search to find a route for the trailer from Nairobi to Kisumu. We are using the towns and distances from Part 1.

UCS looks at the total distance travelled so far. It always picks the town that can be reached at the lowest cost, then checks the roads from there. We continue until Kisumu is picked from the frontier.

For this exercise, we assume the roads in our graph can be used in both directions and have the same distance each way. Our result only covers the roads we included in Part 1.

#### 1. Set up the problem

Our starting town is Nairobi, and our goal is Kisumu. We treat each town as a state. Travelling on a road takes the trailer from one town to another, and the distance travelled is the cost.

We use UCS because the road distances are different. It keeps adding the road distances together and chooses the route with the smallest total so far.

#### 2. Enter the roads and build the network

Here we put the roads from Part 1 into a list with their distances in kilometres. We only list each road once. Since we assumed that the roads work in both directions, the code adds a connection going each way.

The output below shows the towns and the roads connected to each one. This helps us check the network before we start searching.

In [1]:
from collections import defaultdict
import heapq
import itertools

# The road connections and distances from Part 1 (kilometres).
roadConnections = [
    ("Nairobi", "Nakuru", 161),
    ("Nairobi", "Nyeri", 151),
    ("Nyeri", "Nyahururu", 104),
    ("Nyahururu", "Nakuru", 64),
    ("Nakuru", "Mau Summit", 62),
    ("Mau Summit", "Kericho", 50),
    ("Kericho", "Kisumu", 84),
    ("Nairobi", "Narok", 141),
    ("Narok", "Bomet", 81),
    ("Bomet", "Kericho", 77),
    ("Bomet", "Kisii", 83),
    ("Kisii", "Kisumu", 114),
]

startTown = "Nairobi"
goalTown = "Kisumu"

# Add every road in both directions, as assumed in Part 1.
roadGraph = defaultdict(list)
for townA, townB, distance in roadConnections:
    roadGraph[townA].append((townB, distance))
    roadGraph[townB].append((townA, distance))

print(f"Starting town: {startTown}")
print(f"Destination:   {goalTown}")
print(f"Roads listed:  {len(roadConnections)}")
print("\nRoad network (distance in km):")
for town in sorted(roadGraph):
    connections = ", ".join(f"{neighbour} ({distance} km)" for neighbour, distance in sorted(roadGraph[town]))
    print(f"{town} ---> {connections}")

Starting town: Nairobi
Destination:   Kisumu
Roads listed:  12

Road network (distance in km):
Bomet ---> Kericho (77 km), Kisii (83 km), Narok (81 km)
Kericho ---> Bomet (77 km), Kisumu (84 km), Mau Summit (50 km)
Kisii ---> Bomet (83 km), Kisumu (114 km)
Kisumu ---> Kericho (84 km), Kisii (114 km)
Mau Summit ---> Kericho (50 km), Nakuru (62 km)
Nairobi ---> Nakuru (161 km), Narok (141 km), Nyeri (151 km)
Nakuru ---> Mau Summit (62 km), Nairobi (161 km), Nyahururu (64 km)
Narok ---> Bomet (81 km), Nairobi (141 km)
Nyahururu ---> Nakuru (64 km), Nyeri (104 km)
Nyeri ---> Nairobi (151 km), Nyahururu (104 km)


#### 3. Write the Uniform Cost Search algorithm

This function is where we do the search. We start Nairobi at 0 km and put it in the frontier. The frontier is the list of towns we can visit next, together with the total distance from Nairobi.

Each time, we take the town with the lowest cost. Then we check its roads and add each road's distance to the cost so far. If we find a cheaper way to a town, we save the new cost and the town we came from. When Kisumu is taken from the frontier, we have found the lowest-cost route in our graph.

In [2]:
def showFrontier(frontier, bestCosts, settledTowns):
    """Show the current best routes waiting to be explored."""
    currentEntries = []
    for pathCost, order, town in frontier:
        # The queue can contain an older entry if we found a cheaper route later.
        if town not in settledTowns and pathCost == bestCosts[town]:
            currentEntries.append((pathCost, order, town))
    currentEntries.sort()
    if not currentEntries:
        return "(empty)"
    return ", ".join(f"{town} ({cost} km)" for cost, order, town in currentEntries)


def reconstructPath(parents, town):
    """Follow the saved previous-town links to rebuild a route."""
    route = []
    while town is not None:
        route.append(town)
        town = parents[town]
    route.reverse()
    return route


def uniformCostSearch(roadGraph, startTown, goalTown):
    tieBreaker = itertools.count()
    frontier = [(0, next(tieBreaker), startTown)]
    bestCosts = {startTown: 0}
    parents = {startTown: None}
    settledTowns = set()
    searchSteps = []
    stepNumber = 0

    while frontier:
        pathCost, order, currentTown = heapq.heappop(frontier)

        # Skip an old queue entry if this town was already reached more cheaply.
        if currentTown in settledTowns or pathCost != bestCosts[currentTown]:
            continue

        stepNumber += 1
        settledTowns.add(currentTown)
        decisions = []

        # We check for the goal when it is removed from the frontier.
        if currentTown == goalTown:
            searchSteps.append({
                "step": stepNumber,
                "town": currentTown,
                "cost": pathCost,
                "route": reconstructPath(parents, currentTown),
                "decisions": ["This is Kisumu, so we stop the search."],
                "frontier": showFrontier(frontier, bestCosts, settledTowns),
                "goal": True,
            })
            return reconstructPath(parents, goalTown), pathCost, searchSteps

        # Check each road leaving the town we just selected.
        for neighbour, roadDistance in sorted(roadGraph[currentTown]):
            newCost = pathCost + roadDistance

            if neighbour in settledTowns:
                decision = f"Ignore {neighbour}; it has already been settled."
            elif newCost < bestCosts.get(neighbour, float("inf")):
                oldCost = bestCosts.get(neighbour)
                bestCosts[neighbour] = newCost
                parents[neighbour] = currentTown
                heapq.heappush(frontier, (newCost, next(tieBreaker), neighbour))
                if oldCost is None:
                    decision = f"Add {neighbour} at {pathCost} + {roadDistance} = {newCost} km."
                else:
                    decision = f"Update {neighbour}: {oldCost} km becomes {newCost} km."
            else:
                decision = f"Keep the previous route to {neighbour}; {newCost} km is not cheaper."

            decisions.append(f"{currentTown} -> {neighbour} ({roadDistance} km): {decision}")

        searchSteps.append({
            "step": stepNumber,
            "town": currentTown,
            "cost": pathCost,
            "route": reconstructPath(parents, currentTown),
            "decisions": decisions,
            "frontier": showFrontier(frontier, bestCosts, settledTowns),
            "goal": False,
        })

    return None, float("inf"), searchSteps

path, totalCost, searchSteps = uniformCostSearch(roadGraph, startTown, goalTown)

#### 4. Run the search and follow the steps

Now we run the search. The output shows which town UCS picks, how much it costs to get there, and what happens when we check the roads from that town. It also shows the towns still waiting in the frontier.

We can follow these steps to see how the search moves from Nairobi towards Kisumu.

In [3]:
print("UNIFORM COST SEARCH")
print(f"Start: {startTown}")
print(f"Goal:  {goalTown}")
print("Starting cost: 0 km")
print(f"Frontier: {startTown} (0 km)\n")

for step in searchSteps:
    print("-" * 72)
    print(f"STEP {step['step']}: select {step['town']} at {step['cost']} km")
    print("Route so far: " + " -> ".join(step["route"]))
    if step["goal"]:
        print("Goal test: this is Kisumu, so we stop.")
    else:
        print("We check the roads leaving this town:")
        for decision in step["decisions"]:
            print("  " + decision)
    print("Frontier after this step: " + step["frontier"])


UNIFORM COST SEARCH
Start: Nairobi
Goal:  Kisumu
Starting cost: 0 km
Frontier: Nairobi (0 km)

------------------------------------------------------------------------
STEP 1: select Nairobi at 0 km
Route so far: Nairobi
We check the roads leaving this town:
  Nairobi -> Nakuru (161 km): Add Nakuru at 0 + 161 = 161 km.
  Nairobi -> Narok (141 km): Add Narok at 0 + 141 = 141 km.
  Nairobi -> Nyeri (151 km): Add Nyeri at 0 + 151 = 151 km.
Frontier after this step: Narok (141 km), Nyeri (151 km), Nakuru (161 km)
------------------------------------------------------------------------
STEP 2: select Narok at 141 km
Route so far: Nairobi -> Narok
We check the roads leaving this town:
  Narok -> Bomet (81 km): Add Bomet at 141 + 81 = 222 km.
  Narok -> Nairobi (141 km): Ignore Nairobi; it has already been settled.
Frontier after this step: Nyeri (151 km), Nakuru (161 km), Bomet (222 km)
------------------------------------------------------------------------
STEP 3: select Nyeri at 151 km
Ro

#### 5. Rebuild the route and calculate its cost

Once UCS reaches Kisumu, we use the saved previous towns to work backwards to Nairobi. We reverse that list so it shows the route in the order the trailer travels.

The next cell adds the distance for each road and keeps a running total.

In [4]:
routeSegments = []
runningCost = 0
for townA, townB in zip(path, path[1:]):
    segmentCost = next(distance for neighbour, distance in roadGraph[townA] if neighbour == townB)
    runningCost += segmentCost
    routeSegments.append((townA, townB, segmentCost, runningCost))

print("Route found:")
print(" -> ".join(path))
print("\nDistance for each road:")
for townA, townB, segmentCost, runningTotal in routeSegments:
    print(f"{townA} -> {townB}: {segmentCost} km (total so far: {runningTotal} km)")
print(f"\nTotal path cost: {runningCost} km")
assert runningCost == totalCost

Route found:
Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu

Distance for each road:
Nairobi -> Nakuru: 161 km (total so far: 161 km)
Nakuru -> Mau Summit: 62 km (total so far: 223 km)
Mau Summit -> Kericho: 50 km (total so far: 273 km)
Kericho -> Kisumu: 84 km (total so far: 357 km)

Total path cost: 357 km


#### 6. What we found

UCS found this route:

**Nairobi -> Nakuru -> Mau Summit -> Kericho -> Kisumu**

The total distance is `161 + 62 + 50 + 84 = 357 km`. Kisumu was selected from the frontier at 357 km, so this is the shortest route among the roads in our graph. We did not include roads that were left out of Part 1.

#### 7. Groupwork evidence

Before we submit, we need to add evidence that our group worked on this assignment. We can put a team photo here or add our meeting minutes as a separate PDF. If we use a photo, everyone who worked on the assignment should be visible.

<!-- Example: ![Our group working on the assignment](team_photo.jpg) -->